# Train Transformers (A2 / B2 / B2-Gen) on KuaiRec big_matrix — Colab GPU
**AdRec-GenAI — RecSys rebuttal / resubmission experiments**

Trains the three SASRec-style transformer variants for **40 epochs** on the **full big_matrix** (7,176 users x 10,728 items), per reviewer feedback (5 epochs / small_matrix was not enough).

**Before running:**
1. Runtime → Change runtime type → **GPU** (T4 is fine, A100 faster).
2. Upload these files to a folder in your Google Drive (default: `MyDrive/AdRec-GenAI-data/`):
   - `big_matrix.csv` (1 GB)
   - `user_features.csv`, `item_categories.csv`
   - `user_llm_embeddings.npy`, `user_llm_embeddings_generative.npy` (from `kuairec/embeddings/`)
3. Make sure the latest code is **pushed to GitHub** (`Tanushree28/AdRec-GenAI`).

**After running:** download `transformer_outputs.zip` (last cell) and unzip it into `LLM-rec/outputs/` in your local repo.

In [ ]:
# ── Cell 1: GPU check + clone repo ────────────────────────────────────────────
import torch, os
assert torch.cuda.is_available(), 'No GPU! Runtime -> Change runtime type -> GPU'
print('GPU:', torch.cuda.get_device_name(0))

%cd /content
!rm -rf /content/AdRec-GenAI
!git clone --depth 1 https://github.com/Tanushree28/AdRec-GenAI.git /content/AdRec-GenAI
%cd /content/AdRec-GenAI

# Hard check — if the clone silently failed, stop now instead of limping on with an empty tree.
assert os.path.isdir('/content/AdRec-GenAI/.git'), (
    'Clone failed — /content/AdRec-GenAI/.git is missing. '
    'Runtime -> Restart session, then Run all from the top.'
)
assert os.path.exists('kuairec/embeddings/item_llm_embeddings.npy'), (
    'Repo cloned but item_llm_embeddings.npy is missing from it — check the repo on GitHub.'
)
print('Clone OK:', os.path.exists('LLM-rec/src/run_all.py'))

!pip install -q pyyaml pandas numpy tqdm tensorboard scikit-learn

In [ ]:
# ── Cell 2: Mount Drive + link data into the repo ─────────────────────────────
import os, shutil
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/AdRec-GenAI-data'  # <-- change if your folder differs

os.makedirs('kuairec/data', exist_ok=True)
os.makedirs('kuairec/embeddings', exist_ok=True)

for f in ['big_matrix.csv', 'user_features.csv', 'item_categories.csv']:
    src = os.path.join(DRIVE_DIR, f)
    dst = os.path.join('kuairec/data', f)
    if os.path.exists(src) and not os.path.exists(dst):
        os.symlink(src, dst)   # symlink: avoids copying 1 GB
    print(f, 'OK' if os.path.exists(dst) else 'MISSING (optional for user/item features)')

for f in ['user_llm_embeddings.npy', 'user_llm_embeddings_generative.npy']:
    src = os.path.join(DRIVE_DIR, f)
    dst = os.path.join('kuairec/embeddings', f)
    if os.path.exists(src):
        shutil.copy(src, dst)
    print(f, 'OK' if os.path.exists(dst) else 'MISSING — required for B2/B2-Gen!')

assert os.path.exists('kuairec/data/big_matrix.csv'), 'big_matrix.csv is required'
assert os.path.exists('kuairec/embeddings/item_llm_embeddings.npy'), 'item embeddings should come with the repo'

In [ ]:
# ── Cell 3: Colab config overrides ────────────────────────────────────────────
# big_matrix has ~10,728 items vs small_matrix's ~3,327 — the transformer loss
# scores the WHOLE catalog per step, so memory scales with batch*seqlen*n_items.
# 3.2x more items means the batch needs to come down from the old 64, not up.
import yaml

TRANSFORMER_EPOCHS = 40
TRANSFORMER_BATCH  = 32    # start here on a 22-24GB card (L4/A100); raise cautiously
                           # if nvidia-smi shows lots of headroom after one batch.

with open('LLM-rec/config/base.yaml') as f:
    cfg = yaml.safe_load(f)
cfg['training']['transformer_num_epochs'] = TRANSFORMER_EPOCHS
cfg['training']['transformer_batch_size'] = TRANSFORMER_BATCH
cfg['training']['device'] = 'cuda'
cfg['data']['matrix'] = 'big'   # force big_matrix even if small_matrix.csv is present
with open('LLM-rec/config/colab.yaml', 'w') as f:
    yaml.safe_dump(cfg, f)
print(yaml.safe_dump(cfg['training']))

In [ ]:
# ── Cell 4: Train A2 — ID-Transformer ─────────────────────────────────────────
!python LLM-rec/src/run_all.py --config LLM-rec/config/colab.yaml --models a2

In [ ]:
# ── Cell 5: Train B2 — LLM-Transformer (raw user embeddings) ──────────────────
!python LLM-rec/src/run_all.py --config LLM-rec/config/colab.yaml --models b2

In [ ]:
# ── Cell 6: Train B2-Gen — LLM-Transformer (generative user embeddings) ───────
# NOTE: for the paper, first regenerate summaries for ALL 7,176 users with
# colab_generate_summaries.ipynb and rebuild user_llm_embeddings_generative.npy.
!python LLM-rec/src/run_all.py --config LLM-rec/config/colab.yaml --models b2_gen

In [ ]:
# ── Cell 7: Zip results for download ──────────────────────────────────────────
# Contains checkpoints, per-epoch metrics, and tensorboard events for all 3 models.
import shutil, os

!rm -rf /content/transformer_outputs /content/transformer_outputs.zip
os.makedirs('/content/transformer_outputs', exist_ok=True)
for sub in ['checkpoints/a2_id_transformer', 'checkpoints/b2_llm_transformer',
            'checkpoints/b2_gen_llm_transformer', 'events', 'logs']:
    src = f'LLM-rec/outputs/{sub}'
    if os.path.exists(src):
        shutil.copytree(src, f'/content/transformer_outputs/{sub}', dirs_exist_ok=True)

shutil.make_archive('/content/transformer_outputs', 'zip', '/content/transformer_outputs')
print('Size:', os.path.getsize('/content/transformer_outputs.zip') // 1024**2, 'MB')

# Also save a copy to Drive in case the download is interrupted
shutil.copy('/content/transformer_outputs.zip', DRIVE_DIR)
from google.colab import files
files.download('/content/transformer_outputs.zip')

## After downloading
Unzip into your local repo so the folder structure matches:
```bash
unzip transformer_outputs.zip -d LLM-rec/outputs/
```
Then the local evaluation scripts (Hit@10, head/tail analysis, significance tests) can load these checkpoints directly.